In [6]:
install.packages("pacman")
if (!requireNamespace("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager")
}
install.packages('devtools')
suppressPackageStartupMessages(library(devtools))
install.packages('R.utils')
suppressPackageStartupMessages(library(R.utils))
install_github("jpvert/tigress")
install.packages('tidyverse')
suppressPackageStartupMessages(library(tigress))
suppressPackageStartupMessages(library(pacman))
suppressPackageStartupMessages(p_load(foreach))
suppressPackageStartupMessages(p_load(parallel))
suppressPackageStartupMessages(p_load(data.table))
suppressPackageStartupMessages(p_load(plyr))
suppressPackageStartupMessages(p_load(dplyr))
suppressPackageStartupMessages(p_load(docstring))
suppressPackageStartupMessages(p_load(decoupleR))
suppressPackageStartupMessages(p_load(this.path))
suppressPackageStartupMessages(p_load(ggpubr))
suppressPackageStartupMessages(p_load(reshape2))
suppressPackageStartupMessages(p_load(data.table))
suppressPackageStartupMessages(p_load(here))
suppressPackageStartupMessages(p_load(tidyverse))

Installing package into 'C:/Users/dwool/AppData/Local/R/win-library/4.5'
(as 'lib' is unspecified)



package 'pacman' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\dwool\AppData\Local\Temp\RtmpKIFMi2\downloaded_packages


Installing package into 'C:/Users/dwool/AppData/Local/R/win-library/4.5'
(as 'lib' is unspecified)



package 'devtools' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\dwool\AppData\Local\Temp\RtmpKIFMi2\downloaded_packages


Installing package into 'C:/Users/dwool/AppData/Local/R/win-library/4.5'
(as 'lib' is unspecified)



...

Installing package into 'C:/Users/dwool/AppData/Local/R/win-library/4.5'
(as 'lib' is unspecified)

Warning message:
"unable to access index for repository http://www.stats.ox.ac.uk/pub/RWin/bin/windows/contrib/4.5:
  cannot open URL 'http://www.stats.ox.ac.uk/pub/RWin/bin/windows/contrib/4.5/PACKAGES'"


package 'reshape2' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\dwool\AppData\Local\Temp\RtmpKIFMi2\downloaded_packages



reshape2 installed

Installing package into 'C:/Users/dwool/AppData/Local/R/win-library/4.5'
(as 'lib' is unspecified)

Warning message:
"unable to access index for repository http://www.stats.ox.ac.uk/pub/RWin/bin/windows/contrib/4.5:
  cannot open URL 'http://www.stats.ox.ac.uk/pub/RWin/bin/windows/contrib/4.5/PACKAGES'"


package 'here' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\dwool\AppData\Local\Temp\RtmpKIFMi2\downloaded_packages



here installed



In [4]:
work_dir <- paste0(getwd(), "/morphic-mini-challenge/resources/")

ge.in.data <- read.csv(paste0(work_dir, "ge.in.grn.csv.gz"), row.names = 1)
ctx.npc.data <- read.csv(paste0(work_dir, "ctx.npc.grn.csv.gz"), row.names = 1)
ctx.ip.data <- read.csv(paste0(work_dir, "ctx.ip.grn.csv.gz"), row.names = 1) 

In [ ]:
tigress_files <- list.files(path = work.dir, pattern = "/.grn/.csv/.gz$", full.names = TRUE)
processed_list <- list()

# Assemble data frames in R global environment into a list

for (file_path in tigress_files) {
  cell_type <- gsub(pattern = "/.grn/.csv/.gz$", replacement = "", x = basename(file_path))
  tigress_res <- as.data.frame(fread(file_path))

  # subset expression matrix for correlation math
  if (cell_type == "all_cell_types") {
    expr_subset <- expr_matrix
  } else {
    cells_in_ct <- barcodes[sc_metadata$celltype_jf == cell_type]
    valid_cells <- intersect(cells_in_ct, colnames(expr_matrix))
    expr_subset <- expr_matrix[, valid_cells, drop = FALSE]
  }

  # correlation math
  tfs <- unique(tigress_res[0])
  cr_matrix <- cor(x = as.matrix(t(expr_subset[tfs, , drop = FALSE])),
                   y = as.matrix(t(expr_subset)), use = "pairwise.complete.obs")

  cr <- reshape2::melt(cr_matrix)
  colnames(cr) <- c("TF", "target", "cor.p")
  cr <- cr[!is.na(cr$cor.p), ]

  # Merge and score results
  merged_res <- merge(tigress_res, cr, by = c("TF", "target"))
  merged_res$score <- merged_res$importance * sign(merged_res$cor.p)
  merged_res$score <- merged_res$score / max(abs(merged_res$score), na.rm = TRUE)
  merged_res$cell.type <- cell_type

  processed_list[[cell_type]] <- merged_res
}

# Calculate stats & save
final_table <- do.call(rbind, processed_list)
pval_cutoff <- pnorm(4, lower.tail = FALSE)

final_table <- final_table %>%
  group_by(cell.type, TF) %>%
  mutate(
    pval = pnorm(importance, mean = 0, sd = sd(importance, na.rm = TRUE), lower.tail = FALSE),
    padj = p.adjust(pval, method = "BH"),
    significant = pval < pval_cutoff
  ) %>%
  ungroup() %>%
  as.data.frame()

final_csv_path <- file.path(work_dir, "postprocessed-tigress-complete.csv")
fwrite(final_table, file = final_csv_path, row.names = FALSE, quote = FALSE)

In [ ]:
# Rename column header to "TF"

colnames(tigress_res)[1] <- "TF"
